source API URL : "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

JSON Target File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/weather-data/
"

In [0]:
#weatherDataSourceAPIURL = "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weatherDataSourceAPIBaseURL = "https://archive-api.open-meteo.com/v1/archive?latitude="
weatherDataSourceAPIURLOptions = "&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weatherDataSinkLayerName = 'bronze'
weatherDataSinkStorageAccountName = 'adldatalakehouseyubai'
weatherDataSinkFolderName = 'weather-data'

weatherDataSinkFolderPath = f"abfss://{weatherDataSinkLayerName}@{weatherDataSinkStorageAccountName}.dfs.core.windows.net/{weatherDataSinkFolderName}"

In [0]:
import requests
import json
import pandas as pds

In [0]:
geoLocationsDF = spark.sql("SELECT latitude,longitude,marketName from pricing_analytics.silver.geo_location_silver LIMIT 100 ")
display(geoLocationsDF.count())

In [0]:
import requests
import json
import time

weatherDataAPIResponseList = []

for geoLocations in geoLocationsDF.collect():

    weatherDataSourceAPIURL = (
        f"{weatherDataSourceAPIBaseURL}"
        f"{geoLocations['latitude']}"
        f"&longitude={geoLocations['longitude']}"
        f"&start_date=2023-01-01"
        f"&end_date=2023-12-31"
        f"{weatherDataSourceAPIURLOptions}"
    )

    try:
        weatherDataAPIResponse = requests.get(
            weatherDataSourceAPIURL,
            timeout=30
        ).json()

        if isinstance(weatherDataAPIResponse, dict):
            weatherDataAPIResponse["marketName"] = geoLocations["marketName"]

            weatherDataAPIResponseList.append(
                json.dumps(weatherDataAPIResponse)
            )

    except requests.exceptions.Timeout:
        print(f"Timeout for market: {geoLocations['marketName']}")

    except requests.exceptions.RequestException as e:
        print(f"Request failed for market: {geoLocations['marketName']}, error: {e}")

    time.sleep(0.5)


tempWeatherDataPath = f"{weatherDataSinkFolderPath}_temp"

dbutils.fs.put(
    tempWeatherDataPath,
    "\n".join(weatherDataAPIResponseList),
    overwrite=True
)

weatherDataSparkDF = spark.read.json(tempWeatherDataPath)

(weatherDataSparkDF
 .write
 .mode("overwrite")
 .json(weatherDataSinkFolderPath))